# 6 - Boolean network sampling: 12 PCW versus 17 PCW (largest SCC only)

Sample independent Boolean network ensembles from the **largest strongly connected component (SCC)** exported for each week by `5-Component_analysis_12w_vs_17w.ipynb`. The current executed component analysis reports **8 genes for 12 PCW** and **10 genes for 17 PCW**. Only these core genes and their signed regulatory edges enter model synthesis.

## 1. Settings and existing component-analysis files

In [1]:
from pathlib import Path
import hashlib
import json
import time

import numpy as np
import pandas as pd
from IPython.display import display

# Per-week sampling settings
WEEKS = ('12w', '17w')
LABELS = {'12w': '12 PCW', '17w': '17 PCW'}
N_MODELS = 50                      # DISTINCT networks PER WEEK
# Limits on proposed networks, not limits on accepted unique networks.
MAX_DIVERSE_ATTEMPTS = 10 * N_MODELS
MAX_ENUMERATION_ATTEMPTS = 20 * N_MODELS
DIVERSE = True                    # BoNesis diversity-oriented heuristic
MAXCLAUSE = 8
PREFLIGHT = True                  # test original dynamics on each reduced network
RUN_ATTRACTOR_ANALYSIS = True

RESULTS = Path('../Results')
ANALYSIS_DIR = RESULTS / 'Component_analysis_Tcell_12w_vs_17w'
SELECTION_DIR = RESULTS / 'Component_selection_Tcell_manual'
ANALYSIS_MANIFEST = ANALYSIS_DIR / 'comparison_manifest.json'
OUTPUT_DIR = RESULTS / 'Boolean_network_sampling_Tcell_12w_vs_17w' / 'core'

if type(N_MODELS) is not int or N_MODELS < 1:
    raise ValueError('N_MODELS must be a positive integer')
if MAXCLAUSE < 1:
    raise ValueError('MAXCLAUSE must be positive')
if MAX_DIVERSE_ATTEMPTS < N_MODELS or MAX_ENUMERATION_ATTEMPTS < N_MODELS:
    raise ValueError('Sampling attempt limits must be at least N_MODELS')
if not ANALYSIS_MANIFEST.is_file():
    raise FileNotFoundError(f'Missing {ANALYSIS_MANIFEST}. Run your current component-analysis notebook first.')

manifest = json.loads(ANALYSIS_MANIFEST.read_text(encoding='utf-8'))
specs = manifest.get('manual_dynamics_from_selection_notebook', {})
if set(WEEKS) - set(specs):
    raise ValueError('Analysis manifest lacks manual dynamics for one or both weeks')

FILES = {}
for week in WEEKS:
    FILES[week] = {
        'sif': ANALYSIS_DIR / week / 'synthesis_core_network.sif',
        'genes': ANALYSIS_DIR / week / 'core_genes.txt',
        'observations': SELECTION_DIR / week / 'candidate_state_observations.csv',
        'candidate_sif': SELECTION_DIR / week / 'candidate_human_DoRothEA.sif',
        'log': SELECTION_DIR / week / 'clingo_optimization.log',
        'out': OUTPUT_DIR / week,
    }
    for kind in ('sif', 'genes', 'observations', 'candidate_sif', 'log'):
        if not FILES[week][kind].is_file():
            raise FileNotFoundError(f'{LABELS[week]}: missing {kind}: {FILES[week][kind]}')

print(f'Network: largest SCC only; {N_MODELS} models PER WEEK; diverse={DIVERSE}')
print('Sampling outputs:', OUTPUT_DIR)

Network: largest SCC only; 50 models PER WEEK; diverse=True
Sampling outputs: ../Results/Boolean_network_sampling_Tcell_12w_vs_17w/core


## 2. Verify the SCC inputs and selection provenance

In [2]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load_signed_sif(path):
    df = pd.read_csv(path, sep=r'\s+', header=None, comment='#', dtype=str)
    if df.shape[1] != 3 or df.empty:
        raise ValueError(f'{path}: expected nonempty, three-column signed SIF')
    df.columns = ['source', 'sign', 'target']
    df = df.apply(lambda col: col.str.strip())
    df[['source', 'target']] = df[['source', 'target']].apply(lambda col: col.str.upper())
    if df.isna().any().any() or not df['sign'].isin(['1', '-1']).all():
        raise ValueError(f'{path}: invalid gene name or edge sign')
    if df.duplicated().any():
        print('Note:', path, 'contains duplicate SIF lines; removing exact duplicates')
        df = df.drop_duplicates()
    return df.reset_index(drop=True)


def gene_list(path):
    genes = [line.strip().upper() for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if len(genes) != len(set(genes)) or not genes:
        raise ValueError(f'{path}: empty or duplicated genes')
    return set(genes)


def check_provenance(week):
    item = manifest.get('weeks', {}).get(week, {})
    if not item:
        raise ValueError(f'{week} missing from comparison_manifest.json')
    required = {
        'log_sha256': 'log',
        'candidate_sif_sha256': 'candidate_sif',
        'observation_sha256': 'observations',
    }
    for hash_key, file_key in required.items():
        expected = item.get(hash_key)
        if not expected:
            raise ValueError(f'{week}: no {hash_key} in analysis manifest')
        if sha256(FILES[week][file_key]) != expected:
            raise ValueError(f'{week}: {file_key} has changed since component analysis. Rerun 5-Component_analysis.')
    text = FILES[week]['log'].read_text(encoding='utf-8', errors='replace')
    if 'OPTIMUM FOUND' not in text:
        raise RuntimeError(f'{week}: the Clingo optimization did not report OPTIMUM FOUND')
    return item.get('clingo_answer_id')

networks = {}
for week in WEEKS:
    answer_id = check_provenance(week)
    genes = gene_list(FILES[week]['genes'])
    edges = load_signed_sif(FILES[week]['sif'])
    included = set(edges.source) | set(edges.target)
    missing = genes - included
    extra = included - genes
    if missing or extra:
        raise ValueError(
            f'{week} largest SCC: SIF/gene-list mismatch. '
            f'Genes without any SIF interaction: {sorted(missing)}; '
            f'unexpected edge genes: {sorted(extra)}. '
            'Check the core network exported by Component Analysis; no SCC gene will be silently removed.'
        )
    networks[week] = {'genes': genes, 'edges': edges, 'clingo_answer_id': answer_id, 'sif_hash': sha256(FILES[week]['sif'])}
    print(f'{LABELS[week]}: {len(genes)} genes, {len(edges)} signed edges; Clingo answer {answer_id}')

# Existing partial ensembles are checked and resumed in Section 6.
# Model files from other datasets or settings are never silently reused.


12 PCW: 8 genes, 21 signed edges; Clingo answer 806
17 PCW: 10 genes, 24 signed edges; Clingo answer 931


## 3. Restrict observations to the SCC genes (0, 1, ND)

In [3]:
def binary_value(value):
    if pd.isna(value):
        return None
    token = str(value).strip().upper()
    if token in ('', 'ND', 'NA', 'NAN', 'NONE', '?'):
        return None
    try:
        number = float(token)
    except ValueError as exc:
        raise ValueError(f'Unexpected binary observation {value!r}') from exc
    if number in (0.0, 1.0):
        return int(number)
    raise ValueError(f'Expected 0, 1 or ND; found {value!r}')


def read_observations(path, modeled_genes):
    df = pd.read_csv(path, index_col=0, dtype=str, keep_default_na=False)
    df.index = df.index.astype(str).str.strip()
    df.columns = df.columns.astype(str).str.strip().str.upper()
    if df.index.duplicated().any() or df.columns.duplicated().any():
        raise ValueError(f'{path}: duplicate state or gene symbol')
    # The CSV is sparse; omitted columns correspond to all-ND genes.
    return {
        state: {
            gene: value for gene in modeled_genes
            if gene in df.columns and (value := binary_value(df.at[state, gene])) is not None
        }
        for state in df.index
    }

observations = {}
observation_rows = []
for week in WEEKS:
    obs = read_observations(FILES[week]['observations'], networks[week]['genes'])
    spec = specs[week]
    if spec.get('root') != 'S3' or spec.get('fixed_points') != ['S4']:
        raise ValueError(f'{week}: expected S3 root and mandatory S4 fixed point; inspect analysis manifest')
    links = [tuple(pair) for pair in spec['positive_reachability']]
    if not links or any(len(pair) != 2 for pair in links):
        raise ValueError(f'{week}: invalid manual positive reachability list')
    required_states = {'S3', 'S4'} | {state for pair in links for state in pair}
    missing = required_states - set(obs)
    if missing:
        raise ValueError(f'{week}: STREAM state observations missing: {sorted(missing)}')
    if len(obs) != {'12w': 8, '17w': 6}[week]:
        raise ValueError(f'{week}: expected {8 if week == "12w" else 6} STREAM states; got {len(obs)}')
    # A completely unconstrained state should not define a dynamical endpoint.
    missing_calls = [state for state in sorted(required_states) if not obs[state]]
    if missing_calls:
        raise ValueError(
            f'{week}: no determined observations on the largest SCC for '
            f'{missing_calls}. Check the core observations; '
            'the required biological constraints will not be removed automatically.'
        )
    observations[week] = obs
    for state in sorted(obs):
        observation_rows.append({
            'week': week, 'state': state, 'modeled_genes': len(networks[week]['genes']),
            'determined_calls': len(obs[state]),
            'undetermined_genes': len(networks[week]['genes'])-len(obs[state]),
            'is_fixed_point': state == 'S4',
            'is_on_required_route': state in required_states,
        })
    print(f'{LABELS[week]}: {len(obs)} STREAM states; determined observations:',
          {state: len(row) for state, row in sorted(obs.items())})

coverage = pd.DataFrame(observation_rows)
display(coverage)

12 PCW: 8 STREAM states; determined observations: {'S0': 7, 'S1': 7, 'S2': 7, 'S3': 7, 'S4': 7, 'S5': 7, 'S6': 7, 'S7': 7}
17 PCW: 6 STREAM states; determined observations: {'S0': 9, 'S1': 9, 'S2': 8, 'S3': 8, 'S4': 9, 'S5': 8}


,week,state,modeled_genes,determined_calls,undetermined_genes,is_fixed_point,is_on_required_route
0,12w,S0,8,7,1,False,True
1,12w,S1,8,7,1,False,True
2,12w,S2,8,7,1,False,True
3,12w,S3,8,7,1,False,True
4,12w,S4,8,7,1,True,True
5,12w,S5,8,7,1,False,False
6,12w,S6,8,7,1,False,True
7,12w,S7,8,7,1,False,True
8,17w,S0,10,9,1,False,True
9,17w,S1,10,9,1,False,True


## 4. Reuse the exact manual dynamics in both independent BoNesis problems

In [4]:
import bonesis


def make_sampling_problem(week):
    graph = bonesis.InfluenceGraph.from_sif(
        str(FILES[week]['sif']),
        maxclause=MAXCLAUSE,
        allow_skipping_nodes=False,   # sample the exact reduced gene set
        canonic=False,
    )
    if set(graph.nodes()) != networks[week]['genes']:
        raise ValueError(f'{week}: BoNesis did not preserve the exported modeled genes')
    bo = bonesis.BoNesis(graph, observations[week])
    # One REQUIRED fixed point, exactly as in the selection notebook.
    s4 = bo.fixed(~bo.obs('S4'))
    for source, target in specs[week]['positive_reachability']:
        # The final transition must reach the *fixed* S4 configuration.
        destination = s4 if target == 'S4' else ~bo.obs(target)
        ~bo.obs(source) >= destination
    return bo

problems = {week: make_sampling_problem(week) for week in WEEKS}
for week in WEEKS:
    print(LABELS[week], 'S4 required fixed point;',
          len(specs[week]['positive_reachability']), 'positive reachabilities.')

ipylab module is not installed, menus and toolbar are disabled.


12 PCW S4 required fixed point; 6 positive reachabilities.
17 PCW S4 required fixed point; 4 positive reachabilities.


## 5. Check the entire manual specification on each SCC

The original BoNesis optimum was computed before the SCC was extracted. Therefore, that optimum does not guarantee that the **SCC alone** can realize the specified transitions or the required S4 fixed point. A satisfiability preflight runs on each reduced SCC; if no model exists, **sampling stops** without changing any biological assumption.


In [5]:
if PREFLIGHT:
    for week in WEEKS:
        print(f'Preflight {LABELS[week]} (largest SCC)...', flush=True)
        first = next(iter(problems[week].boolean_networks(limit=1)), None)
        if first is None:
            raise RuntimeError(
                f'{LABELS[week]} largest SCC has NO model satisfying '
                'the required S4 fixed point and positive reachability constraints. '
                'This SCC is incompatible with the specified dynamics; sampling stopped. '
                'Review the SCC edges and binary observations. The constraints were NOT relaxed.'
            )
        if set(first) != networks[week]['genes']:
            raise RuntimeError(f'{week}: preflight returned unexpected model genes')
        print(f'{LABELS[week]}: at least one full-specification model exists.')
else:
    print('Preflight disabled. Incompatibility may only become clear at sampling time.')

Preflight 12 PCW (largest SCC)...
Grounding...done in 0.0s
12 PCW: at least one full-specification model exists.
Preflight 17 PCW (largest SCC)...
Grounding...done in 0.0s
17 PCW: at least one full-specification model exists.


## 6. Sample **distinct** Boolean networks from each SCC

BoNesis diversity-oriented enumeration is a heuristic, **not a uniqueness guarantee**. A proposed Boolean network can match an earlier one exactly, especially for a small SCC with stringent dynamical constraints. Skip duplicate saved `.bnet` files, try additional proposals, and switch to ordinary BoNesis enumeration if the diversity-oriented pass finds fewer than 50 unique networks. 

The search is bounded by `MAX_DIVERSE_ATTEMPTS` and `MAX_ENUMERATION_ATTEMPTS` to avoid unlimited repeats. If fewer than 50 distinct networks are found, preserve the partial ensemble and report the actual count. 

In [6]:
def expected_run_properties(week):
    return {
        'week': week,
        'network_level': 'largest_scc',
        'number_requested': N_MODELS,
        'diverse_sampling': DIVERSE,
        'maxclause': MAXCLAUSE,
        'modeled_gene_count': len(networks[week]['genes']),
        'signed_edge_count': len(networks[week]['edges']),
        'clingo_optimal_answer_id': networks[week]['clingo_answer_id'],
        'input_sif_sha256': networks[week]['sif_hash'],
        'input_observations_sha256': sha256(FILES[week]['observations']),
        'required_fixed_points': ['S4'],
        'positive_reachability': specs[week]['positive_reachability'],
        'negative_reachability': [],
    }


MANIFEST_COLUMNS = ['week', 'model_id', 'filename', 'sha256', 'size_bytes', 'sampling_method']


def load_previous_models(week, out, run_properties):
    """Resume only a verified ensemble generated for identical inputs/constraints."""
    paths = sorted(out.glob('bn[0-9][0-9][0-9][0-9].bnet'))
    manifest_path = out / 'model_manifest.csv'
    metadata_path = out / 'run_metadata.json'
    if not paths and not manifest_path.exists() and not metadata_path.exists():
        return []
    if not manifest_path.is_file() or not metadata_path.is_file():
        raise RuntimeError(f'{week}: existing models or audit files are incomplete in {out}. '
                           'Archive this directory before starting a new ensemble.')
    old_info = json.loads(metadata_path.read_text(encoding='utf-8'))
    for key, expected in run_properties.items():
        if old_info.get(key) != expected:
            raise RuntimeError(f'{week}: existing ensemble has different {key}. '
                               'Archive this output directory before starting a new analysis.')
    frame = pd.read_csv(manifest_path)
    required = {'week', 'model_id', 'filename', 'sha256', 'size_bytes'}
    if not required.issubset(frame.columns):
        raise RuntimeError(f'{week}: cannot validate the old model manifest.')
    records = frame.to_dict('records')
    if len(paths) != len(records):
        raise RuntimeError(f'{week}: manifest lists {len(records)} files but found {len(paths)}. '
                           'Restore the files or archive the directory.')
    seen_hashes = set()
    for j, (row, path) in enumerate(zip(records, paths)):
        expected_name = f'bn{j:04d}.bnet'
        if (str(row['week']) != week or int(row['model_id']) != j
                or row['filename'] != expected_name or path.name != expected_name):
            raise RuntimeError(f'{week}: existing filenames or model indices are inconsistent.')
        actual_hash = sha256(path)
        if row['sha256'] != actual_hash or int(row['size_bytes']) != path.stat().st_size:
            raise RuntimeError(f'{week}: saved file {path.name} differs from its manifest.')
        if actual_hash in seen_hashes:
            raise RuntimeError(f'{week}: existing files contain duplicate models.')
        seen_hashes.add(actual_hash)
        row['sampling_method'] = row.get('sampling_method', 'previous_run')
    if len(records) > N_MODELS:
        raise RuntimeError(f'{week}: previous ensemble contains more than N_MODELS.')
    return records


def write_sampling_checkpoint(week, out, saved, run_properties, started,
                              attempts_diverse, attempts_enum, duplicates, previously_saved):
    """Persist recovery information whenever the ensemble changes."""
    df = pd.DataFrame(saved, columns=MANIFEST_COLUMNS)
    tmp_manifest = out / 'model_manifest.csv.tmp'
    df.to_csv(tmp_manifest, index=False)
    tmp_manifest.replace(out / 'model_manifest.csv')
    data = {
        **run_properties,
        'label': LABELS[week],
        'status': 'complete' if len(saved) == N_MODELS else 'incomplete',
        'number_saved': len(saved),
        'resumed_from_existing': previously_saved,
        'diverse_proposals_this_run': attempts_diverse,
        'enumeration_proposals_this_run': attempts_enum,
        'duplicates_skipped_this_run': duplicates,
        'max_diverse_attempts': MAX_DIVERSE_ATTEMPTS,
        'max_enumeration_attempts': MAX_ENUMERATION_ATTEMPTS,
        'preflight_passed': PREFLIGHT,
        'elapsed_seconds_this_run': round(time.perf_counter() - started, 2),
    }
    tmp_meta = out / 'run_metadata.json.tmp'
    tmp_meta.write_text(json.dumps(data, indent=2), encoding='utf-8')
    tmp_meta.replace(out / 'run_metadata.json')
    return data


sampled = {}
all_run_metadata = {}
for week in WEEKS:
    out = FILES[week]['out']
    out.mkdir(parents=True, exist_ok=True)
    run_properties = expected_run_properties(week)
    saved = load_previous_models(week, out, run_properties)
    existing = len(saved)
    seen = {item['sha256'] for item in saved}
    attempts_diverse = 0
    attempts_enum = 0
    duplicates = 0
    started = time.perf_counter()

    # These files depend only on the validated biological inputs, not on sampling.
    pd.DataFrame.from_dict(observations[week], orient='index').rename_axis('state').to_csv(
        out / 'observations_used.csv', na_rep='ND'
    )
    pd.DataFrame([r for r in observation_rows if r['week'] == week]).to_csv(
        out / 'observation_coverage.csv', index=False
    )
    (out / 'dynamics_used.json').write_text(json.dumps(specs[week], indent=2), encoding='utf-8')
    (out / 'modeled_genes.txt').write_text('\n'.join(sorted(networks[week]['genes'])) + '\n')
    print(f'{LABELS[week]}: {existing} previously saved unique models; target {N_MODELS}', flush=True)

    # The proposal filename is NEVER a numbered model and is removed on every pass.
    proposal_path = out / '_candidate_model.bnet'
    if proposal_path.exists():
        proposal_path.unlink()
    try:
        if existing < N_MODELS:
            methods = []
            if DIVERSE:
                methods.append(('diverse', MAX_DIVERSE_ATTEMPTS,
                                lambda limit: problems[week].diverse_boolean_networks(limit=limit)))
            # The ordinary enumeration pass provides a second route to unique solutions.
            methods.append(('enumeration', MAX_ENUMERATION_ATTEMPTS,
                            lambda limit: problems[week].boolean_networks(limit=limit)))
            for method, budget, make_view in methods:
                if len(saved) == N_MODELS:
                    break
                print(f'{LABELS[week]}: starting {method}; up to {budget} proposals', flush=True)
                for bn in make_view(budget):
                    if method == 'diverse':
                        attempts_diverse += 1
                    else:
                        attempts_enum += 1
                    if set(bn) != networks[week]['genes']:
                        raise RuntimeError(f'{week}: proposed network has unexpected nodes')
                    bn.save(str(proposal_path))
                    digest = sha256(proposal_path)
                    if digest in seen:
                        duplicates += 1
                        proposal_path.unlink(missing_ok=True)
                        continue
                    next_id = len(saved)
                    dest = out / f'bn{next_id:04d}.bnet'
                    if dest.exists():
                        raise RuntimeError(f'{week}: refusing to overwrite unexpected file {dest}')
                    proposal_path.replace(dest)
                    seen.add(digest)
                    saved.append({
                        'week': week,
                        'model_id': next_id,
                        'filename': dest.name,
                        'sha256': digest,
                        'size_bytes': dest.stat().st_size,
                        'sampling_method': method,
                    })
                    write_sampling_checkpoint(week, out, saved, run_properties, started,
                                              attempts_diverse, attempts_enum, duplicates, existing)
                    if len(saved) % 10 == 0 or len(saved) == N_MODELS:
                        print(f'{LABELS[week]}: {len(saved)}/{N_MODELS} distinct models saved', flush=True)
                    if len(saved) == N_MODELS:
                        break
    finally:
        proposal_path.unlink(missing_ok=True)
        data = write_sampling_checkpoint(week, out, saved, run_properties, started,
                                         attempts_diverse, attempts_enum, duplicates, existing)
        sampled[week] = saved
        all_run_metadata[week] = data
    if len(saved) < N_MODELS:
        raise RuntimeError(
            f'{week}: only {len(saved)} DISTINCT models after '
            f'{attempts_diverse} diverse proposals + {attempts_enum} enumeration proposals '
            f'({duplicates} duplicate proposals discarded). '
            f'The partial ensemble is preserved in {out}. '
            'This is NOT proof that no other compatible models exist. '
            'Increase the attempt limits or investigate the constraints before drawing conclusions.'
        )

print('COMPLETE: separate unique-model ensembles sampled for both weeks.')

12 PCW: 0 previously saved unique models; target 50
12 PCW: starting diverse; up to 500 proposals
Grounding...done in 0.0s
Found 23/500 solutions in 0.1s (first in 0.0s; rate 0.0s)12 PCW: 10/50 distinct models saved
Found 84/500 solutions in 0.8s (first in 0.0s; rate 0.0s)12 PCW: 20/50 distinct models saved
Found 128/500 solutions in 1.9s (first in 0.0s; rate 0.0s)12 PCW: 30/50 distinct models saved
Found 218/500 solutions in 5.6s (first in 0.0s; rate 0.0s)12 PCW: 40/50 distinct models saved
Found 287/500 solutions in 10.4s (first in 0.0s; rate 0.0s)12 PCW: 50/50 distinct models saved
17 PCW: 0 previously saved unique models; target 50
17 PCW: starting diverse; up to 500 proposals
Grounding...done in 0.0s
Found 11/500 solutions in 0.1s (first in 0.0s; rate 0.0s)17 PCW: 10/50 distinct models saved
Found 24/500 solutions in 0.2s (first in 0.0s; rate 0.0s)17 PCW: 20/50 distinct models saved
Found 38/500 solutions in 0.4s (first in 0.0s; rate 0.0s)17 PCW: 30/50 distinct models saved
Found 

## 7. Boolean rule variability within each week

The following counts how many *distinct textual Boolean expressions* were sampled for each gene. These counts are a descriptive lower-level proxy for rule diversity: syntactically different expressions may encode logically equivalent functions. Gene-to-gene comparisons should account for different input GRNs and candidate domains.

In [8]:
def read_bnet_rules(path):
    rules = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        name, sep, expr = line.partition(',')
        if not sep or not name.strip() or not expr.strip():
            raise ValueError(f'{path}: malformed .bnet rule line: {line[:100]}')
        name = name.strip().upper()
        if name in rules:
            raise ValueError(f'{path}: duplicated gene {name}')
        rules[name] = ''.join(expr.split())  # ignore whitespace only
    return rules

rule_rows = []
for week in WEEKS:
    rule_sets = {gene: [] for gene in sorted(networks[week]['genes'])}
    for item in sampled[week]:
        rules = read_bnet_rules(FILES[week]['out'] / item['filename'])
        if set(rules) != networks[week]['genes']:
            raise ValueError(f'{week} {item["filename"]}: genes do not match exported SIF')
        for gene, rule in rules.items():
            rule_sets[gene].append(rule)
    for gene, expressions in rule_sets.items():
        counts = pd.Series(expressions).value_counts()
        rule_rows.append({
            'week': week, 'gene': gene, 'n_models': len(expressions),
            'distinct_textual_rules': len(counts),
            'most_frequent_rule_fraction': float(counts.iloc[0] / len(expressions)),
            'most_frequent_rule': counts.index[0],
        })
    pd.DataFrame([r for r in rule_rows if r['week'] == week]).to_csv(
        FILES[week]['out'] / 'boolean_rule_variability.csv', index=False
    )

rule_df = pd.DataFrame(rule_rows)
rule_df.to_csv(OUTPUT_DIR / 'rule_variability_12w_vs_17w.csv', index=False)
rule_summary = rule_df.groupby('week', as_index=False).agg(
    sampled_genes=('gene', 'count'),
    mean_distinct_textual_rules=('distinct_textual_rules', 'mean'),
    genes_with_one_textual_rule=('distinct_textual_rules', lambda x: int((x == 1).sum())),
)
display(rule_summary)

,week,sampled_genes,mean_distinct_textual_rules,genes_with_one_textual_rule
0,12w,8,3.0,2
1,17w,10,1.9,3


## 9. Final sampling summary 

In [9]:
summary = pd.DataFrame([{
    'week': week, 'network_level': 'largest_scc',
    'modeled_genes': len(networks[week]['genes']),
    'signed_regulations': len(networks[week]['edges']),
    'stream_states': len(observations[week]),
    'positive_reachabilities': len(specs[week]['positive_reachability']),
    'S4_is_required_fixed_point': True,
    'requested_models': N_MODELS,
    'saved_models': len(sampled[week]),
    'optimization_proven_in_original_candidate_domain': True,
    'reduced_domain_preflight_checked': PREFLIGHT,
    'complete': len(sampled[week]) == N_MODELS,
} for week in WEEKS])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_DIR / 'sampling_summary_12w_vs_17w.csv', index=False)
display(summary)
print('Models, audits, and comparisons saved under:', OUTPUT_DIR.resolve())

,week,network_level,modeled_genes,signed_regulations,stream_states,positive_reachabilities,S4_is_required_fixed_point,requested_models,saved_models,optimization_proven_in_original_candidate_domain,reduced_domain_preflight_checked,complete
0,12w,largest_scc,8,21,8,6,True,50,50,True,True,True
1,17w,largest_scc,10,24,6,4,True,50,50,True,True,True


Models, audits, and comparisons saved under: /home/eren/Eren_run/Case_study_2/Results/Boolean_network_sampling_Tcell_12w_vs_17w/core
